# Project 7: Compressed Sensing for Signal Recovery

## Introduction

**Compressed Sensing** is a revolutionary technique that recovers signals from far fewer measurements than traditionally required - breaking the Nyquist-Shannon sampling theorem!

### The Compressed Sensing Problem

Given:
- Unknown sparse signal $x \in \mathbb{R}^n$ (most entries are zero)
- Measurement matrix $A \in \mathbb{R}^{m \times n}$ with $m \ll n$
- Measurements $y = Ax \in \mathbb{R}^m$

Recover $x$ from $y$ by solving:

$$\min_x \|x\|_1 \quad \text{subject to} \quad Ax = y$$

### Why L1 Minimization?

- **L0 norm** ($\|x\|_0$ = # non-zeros) is ideal but **NP-hard**
- **L1 norm** ($\|x\|_1 = \sum_i |x_i|$) is **convex relaxation** → solvable!
- Under right conditions (RIP), L1 recovers the same solution as L0

### Applications

- **Medical imaging**: MRI with fewer scans
- **Radar**: Detect targets with less data
- **Photography**: Single-pixel cameras
- **Signal processing**: Spectrum sensing

### Learning Objectives

1. Understand compressed sensing theory
2. Implement signal recovery with CVXPy
3. Apply to 1D sparse signals
4. Recover images from incomplete measurements
5. Understand measurement requirements

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from scipy.fftpack import dct, idct
import seaborn as sns

np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('whitegrid')

## Part 1: Sparse Signal Recovery (1D)

In [ ]:
# Create sparse signal
n = 200  # Signal dimension
k = 10   # Sparsity (# non-zero elements)

# True sparse signal
x_true = np.zeros(n)
nonzero_idx = np.random.choice(n, k, replace=False)
x_true[nonzero_idx] = np.random.randn(k) * 2

print(f"Signal dimension: {n}")
print(f"Sparsity: {k} ({k/n*100:.1f}%)")
print(f"Non-zero indices: {sorted(nonzero_idx)}")

# Visualize true signal
plt.figure(figsize=(14, 4))
plt.stem(range(n), x_true, basefmt=' ', use_line_collection=True)
plt.xlabel('Index', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.title('True Sparse Signal', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: Random Measurements

In [ ]:
# Create measurement matrix (Gaussian random)
m = 60  # Number of measurements (m << n)
A = np.random.randn(m, n) / np.sqrt(m)  # Normalized

# Take measurements
y = A @ x_true

print(f"\n=== Compressed Sensing Setup ===")
print(f"Signal dimension (n): {n}")
print(f"Measurements (m): {m}")
print(f"Compression ratio: {n/m:.1f}x")
print(f"Sparsity (k): {k}")
print(f"\nTheoretical requirement: m ≥ O(k log(n/k)) ≈ {int(k * np.log(n/k) * 2)}")
print(f"We have: {m} measurements")
print(f"Status: {'✓ Sufficient' if m >= k * np.log(n/k) * 2 else '✗ May be insufficient'}")

# Visualize measurement matrix
plt.figure(figsize=(10, 4))
plt.imshow(A, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
plt.colorbar(label='Value')
plt.xlabel('Signal Index (n)', fontsize=11)
plt.ylabel('Measurement Index (m)', fontsize=11)
plt.title('Random Measurement Matrix A', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: Recover Signal with L1 Minimization

In [ ]:
def compressed_sensing_l1(A, y):
    """
    Recover sparse signal using L1 minimization.
    
    min ||x||_1
    s.t. Ax = y
    """
    n = A.shape[1]
    x = cp.Variable(n)
    
    objective = cp.Minimize(cp.norm(x, 1))
    constraints = [A @ x == y]
    
    problem = cp.Problem(objective, constraints)
    problem.solve()
    
    return x.value

# Recover signal
print("Solving L1 minimization...")
x_recovered = compressed_sensing_l1(A, y)

# Compare
recovery_error = np.linalg.norm(x_true - x_recovered) / np.linalg.norm(x_true)
print(f"\nRelative recovery error: {recovery_error:.6f}")
print(f"True sparsity: {np.sum(np.abs(x_true) > 1e-6)}")
print(f"Recovered sparsity: {np.sum(np.abs(x_recovered) > 1e-3)}")

if recovery_error < 0.01:
    print("\n✓ Perfect recovery!")
elif recovery_error < 0.1:
    print("\n✓ Excellent recovery!")
else:
    print("\n✗ Recovery failed. Try more measurements.")

## Part 4: Visualize Recovery

In [ ]:
# Plot comparison
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# True signal
axes[0].stem(range(n), x_true, basefmt=' ', use_line_collection=True, linefmt='C0-', markerfmt='C0o')
axes[0].set_ylabel('Amplitude', fontsize=11)
axes[0].set_title('True Sparse Signal', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

# Recovered signal
axes[1].stem(range(n), x_recovered, basefmt=' ', use_line_collection=True, linefmt='C1-', markerfmt='C1o')
axes[1].set_ylabel('Amplitude', fontsize=11)
axes[1].set_title('Recovered Signal (L1 Minimization)', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

# Error
error = x_true - x_recovered
axes[2].stem(range(n), error, basefmt=' ', use_line_collection=True, linefmt='C3-', markerfmt='C3o')
axes[2].set_xlabel('Index', fontsize=11)
axes[2].set_ylabel('Error', fontsize=11)
axes[2].set_title(f'Recovery Error (Relative: {recovery_error:.4f})', fontsize=13, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: Effect of Number of Measurements

In [ ]:
# Test different numbers of measurements
m_values = [20, 30, 40, 50, 60, 80, 100, 120]
errors = []

for m_test in m_values:
    # Create measurement matrix
    A_test = np.random.randn(m_test, n) / np.sqrt(m_test)
    y_test = A_test @ x_true
    
    # Recover
    try:
        x_rec = compressed_sensing_l1(A_test, y_test)
        error = np.linalg.norm(x_true - x_rec) / np.linalg.norm(x_true)
        errors.append(error)
    except:
        errors.append(np.nan)
    
    print(f"m={m_test:3d}, error={errors[-1]:.6f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Error vs measurements
ax1.plot(m_values, errors, 'o-', linewidth=2, markersize=8)
ax1.axvline(k * np.log(n/k) * 2, color='red', linestyle='--', label=f'Theory: m ≥ {int(k * np.log(n/k) * 2)}')
ax1.axhline(0.01, color='green', linestyle='--', alpha=0.5, label='1% error')
ax1.set_xlabel('Number of Measurements (m)', fontsize=12)
ax1.set_ylabel('Relative Recovery Error', fontsize=12)
ax1.set_title('Recovery Quality vs Measurements', fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend()
ax1.grid(alpha=0.3)

# Compression ratio vs error
compression_ratios = [n/m for m in m_values]
ax2.plot(compression_ratios, errors, 'o-', linewidth=2, markersize=8, color='orange')
ax2.axhline(0.01, color='green', linestyle='--', alpha=0.5, label='1% error')
ax2.set_xlabel('Compression Ratio (n/m)', fontsize=12)
ax2.set_ylabel('Relative Recovery Error', fontsize=12)
ax2.set_title('Recovery vs Compression', fontsize=14, fontweight='bold')
ax2.set_yscale('log')
ax2.invert_xaxis()  # Higher compression on left
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6: Image Recovery from Random Samples

In [ ]:
# Create simple test image
img_size = 32
img = np.zeros((img_size, img_size))

# Add some structure (sparse in pixel domain)
img[10:12, 5:28] = 1.0  # Horizontal line
img[5:28, 15:17] = 1.0  # Vertical line
img[20:25, 20:25] = 0.8  # Square

# Flatten
x_img = img.flatten()
n_img = len(x_img)

print(f"Image size: {img_size}x{img_size} = {n_img} pixels")
print(f"Sparsity: {np.sum(x_img > 0)} non-zero pixels ({np.sum(x_img > 0)/n_img*100:.1f}%)")

# Visualize original
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray', interpolation='nearest')
plt.title('Original Image', fontsize=14, fontweight='bold')
plt.colorbar()
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Random measurements
m_img = int(n_img * 0.3)  # Sample 30% of pixels
A_img = np.random.randn(m_img, n_img) / np.sqrt(m_img)
y_img = A_img @ x_img

print(f"Taking {m_img} random measurements ({m_img/n_img*100:.1f}% of pixels)")
print(f"Compression: {n_img/m_img:.1f}x")

# Recover using L1 minimization
print("\nRecovering image with compressed sensing...")
x_img_recovered = compressed_sensing_l1(A_img, y_img)
img_recovered = x_img_recovered.reshape(img_size, img_size)

# Compute error
recovery_error_img = np.linalg.norm(img - img_recovered) / np.linalg.norm(img)
print(f"Recovery error: {recovery_error_img:.4f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im0 = axes[0].imshow(img, cmap='gray', interpolation='nearest', vmin=0, vmax=1)
axes[0].set_title('Original Image', fontsize=13, fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(img_recovered, cmap='gray', interpolation='nearest', vmin=0, vmax=1)
axes[1].set_title(f'Recovered (30% measurements)', fontsize=13, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

error_img = np.abs(img - img_recovered)
im2 = axes[2].imshow(error_img, cmap='hot', interpolation='nearest')
axes[2].set_title(f'Absolute Error (Rel: {recovery_error_img:.3f})', fontsize=13, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

## Part 7: Basis Pursuit Denoising (with Noise)

In [ ]:
def basis_pursuit_denoising(A, y, epsilon):
    """
    Compressed sensing with noisy measurements.
    
    min ||x||_1
    s.t. ||Ax - y||_2 <= epsilon
    """
    n = A.shape[1]
    x = cp.Variable(n)
    
    objective = cp.Minimize(cp.norm(x, 1))
    constraints = [cp.norm(A @ x - y, 2) <= epsilon]
    
    problem = cp.Problem(objective, constraints)
    problem.solve()
    
    return x.value

# Add noise to measurements
noise_level = 0.05
noise = np.random.randn(m) * noise_level * np.linalg.norm(y) / np.sqrt(m)
y_noisy = y + noise

print(f"\n=== Noisy Compressed Sensing ===")
print(f"Noise level: {noise_level*100:.1f}% of signal norm")
print(f"SNR: {20*np.log10(np.linalg.norm(y)/np.linalg.norm(noise)):.1f} dB")

# Recover with exact constraint (will fail gracefully)
print("\nTrying exact recovery (will be poor with noise)...")
try:
    x_rec_exact = compressed_sensing_l1(A, y_noisy)
    error_exact = np.linalg.norm(x_true - x_rec_exact) / np.linalg.norm(x_true)
    print(f"Error with exact constraint: {error_exact:.4f}")
except:
    print("Failed (infeasible)")
    x_rec_exact = None

# Recover with relaxed constraint
epsilon = np.linalg.norm(noise) * 1.5
print(f"\nUsing basis pursuit denoising with ε = {epsilon:.4f}...")
x_rec_bpd = basis_pursuit_denoising(A, y_noisy, epsilon)
error_bpd = np.linalg.norm(x_true - x_rec_bpd) / np.linalg.norm(x_true)
print(f"Error with BPDN: {error_bpd:.4f}")

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# True vs noisy recovery
if x_rec_exact is not None:
    axes[0].stem(range(n), x_rec_exact, basefmt=' ', use_line_collection=True, 
                linefmt='C1-', markerfmt='C1o', label='Exact constraint')
axes[0].stem(range(n), x_true, basefmt=' ', use_line_collection=True, 
            linefmt='C0-', markerfmt='C0o', label='True', alpha=0.5)
axes[0].set_ylabel('Amplitude', fontsize=11)
axes[0].set_title('Recovery with Exact Constraint (Poor)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# BPDN recovery
axes[1].stem(range(n), x_rec_bpd, basefmt=' ', use_line_collection=True, 
            linefmt='C2-', markerfmt='C2o', label='BPDN')
axes[1].stem(range(n), x_true, basefmt=' ', use_line_collection=True, 
            linefmt='C0-', markerfmt='C0o', label='True', alpha=0.5)
axes[1].set_xlabel('Index', fontsize=11)
axes[1].set_ylabel('Amplitude', fontsize=11)
axes[1].set_title(f'Recovery with BPDN (Better, Error: {error_bpd:.4f})', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

### What We Learned

1. **Compressed sensing** recovers sparse signals from few measurements
2. **L1 minimization** is convex relaxation of L0 (sparsity)
3. **Theoretical requirement**: $m \geq O(k \log(n/k))$ measurements
4. **Random measurements** (Gaussian, Bernoulli) work well
5. **Basis Pursuit Denoising** handles noisy measurements

### Key Concepts

- **Sparsity**: Most signal values are zero
- **Incoherence**: Measurement basis and sparsity basis are different
- **RIP (Restricted Isometry Property)**: Guarantees L1 = L0 solution
- **Underdetermined system**: More unknowns than equations (m < n)

### Real-World Applications

1. **Medical Imaging (MRI)**
   - Reduce scan time from hours to minutes
   - Less radiation exposure
   - Images sparse in wavelet domain

2. **Radar and Sonar**
   - Detect sparse targets
   - Reduce sampling requirements

3. **Single-Pixel Camera**
   - Capture images with one sensor
   - Random projections

4. **Wireless Communication**
   - Spectrum sensing
   - Channel estimation

5. **Machine Learning**
   - LASSO regression (Project 4!)
   - Feature selection
   - Neural network pruning

### Connection to Other Projects

- **Project 4 (LASSO)**: Same L1 minimization principle!
  - LASSO: $\min \|y - Ax\|_2^2 + \lambda\|x\|_1$
  - CS: $\min \|x\|_1$ s.t. $Ax = y$
  - Both encourage sparsity

- **Project 1 (SVD)**: Low-rank approximation vs sparsity
- **Project 2 (PCA)**: Dimensionality reduction

### When Does Compressed Sensing Work?

✅ **Works when:**
- Signal is sparse (or compressible)
- Measurements are incoherent (random)
- Have $m \geq O(k \log(n/k))$ measurements

❌ **Doesn't work when:**
- Signal is dense (not sparse)
- Measurements are coherent with sparsity basis
- Too few measurements

### Extensions

- **Matrix completion**: Netflix problem (low-rank matrix from few entries)
- **Total variation**: Sparsity in gradients (image denoising)
- **Dictionary learning**: Learn sparsity basis from data
- **Deep compressed sensing**: Use neural networks

### Further Reading

- **Classic paper**: Candès & Tao, "Decoding by Linear Programming"
- **Book**: "A Mathematical Introduction to Compressive Sensing" by Foucart & Rauhut
- **Applications**: Lustig et al., "Sparse MRI"

---

## Congratulations!

You've completed all 7 projects of the EECS 127 Crash Course! 🎉

You now understand:
- Matrix decompositions (SVD, PCA)
- Optimization (gradient descent, convex optimization)
- Regularization (L1, L2)
- Quadratic programming (portfolio optimization)
- Compressed sensing

**Next steps:**
- Apply these techniques to your ML projects
- Take Stanford CS 229 (Machine Learning)
- Read Boyd & Vandenberghe's "Convex Optimization"
- Implement modern optimizers (Adam, RMSprop)

**You're now equipped to tackle real-world optimization problems in ML and beyond!** 🚀💪